In [ ]:
import numpy as np, torch, matplotlib.pyplot as plt
import matplotlib.patches as mpatches, matplotlib.colors as mcolors
import rasterio
from rasterio.transform import Affine
from pathlib import Path
from config import TrainConfig, ModelConfig
from model_config import make_model
from inference import FloodInference
from inference import build_subtitle, compute_class_pcts, save_result_as_tif, visualize_result_tif, _load_barangay_band_from_geojson, run_and_export

cfg  = TrainConfig()
mcfg = ModelConfig()
PATCH_SIZE = mcfg.patch_size

# ── Visual setup ──────────────────────────────────────────────────────────────
FLOOD_COLORS = {
     0: '#FFFFFF',
     1: '#C6DBEF',
     2: '#6BAED6',
     3: '#2171B5',
     4: '#08306B',
}
FLOOD_LABELS = {0: 'No Flood', 1: 'Light', 2: 'Moderate', 3: 'Heavy', 4: 'Extreme'}

conf_cmap  = 'RdYlGn'
flood_colors_rgba = [(1.0, 1.0, 1.0, 0.0), mcolors.to_rgba('#C6DBEF'), mcolors.to_rgba('#6BAED6'),  mcolors.to_rgba('#2171B5'),  mcolors.to_rgba('#08306B'), ]
flood_cmap = mcolors.ListedColormap(flood_colors_rgba)
flood_norm = mcolors.BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5, 4.5], ncolors=5)

legend_patches = [
    mpatches.Patch(facecolor='none', edgecolor='gray', linewidth=1.0, linestyle='--', label='Class 0 — No Flood'),
    *[
        mpatches.Patch(facecolor=FLOOD_COLORS[i], edgecolor='gray',
                       linewidth=0.5, label=f'Class {i} — {FLOOD_LABELS[i]}')
        for i in range(1, 5)
    ]
]

# ── Spatial data ──────────────────────────────────────────────────────────────
data                 = np.load(cfg.output_dir / 'spatial_data.npz')
X_full               = data['spatial']
manila_patch_indices = data['manila_patch_indices']
rain_min, rain_max   = float(data['rain_min'].item()), float(data['rain_max'].item())
N_H, N_W             = int(data['n_h'].item()),        int(data['n_w'].item())

# ── GeoTIFF transform ────────────────────────────────────────────────────────
with rasterio.open('COP-30m-GMM/greater_mm_bbox_dem_cop.tif') as src:
    t = src.transform
    dem_crs = src.crs
dem_transform = Affine(t.a * PATCH_SIZE, 0, t.c, 0, t.e * PATCH_SIZE, t.f)

# ── Model ─────────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = make_model(ModelConfig()).to(device)

ckpt_files = sorted(cfg.output_dir.glob('fold_*/best_model.pt'))
best_ckpt = max(ckpt_files, key=lambda f: torch.load(f, map_location='cpu', weights_only=False).get('best_monitor', -np.inf))

ckpt = torch.load(best_ckpt, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

# ── Inference engine ──────────────────────────────────────────────────────────
engine = FloodInference(
    model       = model,
    device      = device,
    rain_min    = rain_min,
    rain_max    = rain_max,
    X_spatial   = X_full,
    num_classes = cfg.num_classes,
    batch_size  = cfg.batch_size,
    manila_patch_indices = manila_patch_indices, 
)

print(f"Grid: {N_H}x{N_W}  |  Transform: {dem_transform.a:.0f}m pixels  |  CRS: {dem_crs}")

In [ ]:
_GEOJSON_PATH  = 'manila_barangay_geojson.geojson'
_barangay_band = _load_barangay_band_from_geojson(_GEOJSON_PATH, dem_transform, dem_crs, N_H, N_W)

In [ ]:
# ── Test configurations ───────────────────────────────────────────────────────
test_configs = [
    {'storm_type': 'triangular',   'depth_mm': 9,  'tpeak': 0.5},
    {'storm_type': 'front-loaded', 'depth_mm': 9,  'tpeak': None},
    {'storm_type': 'back-loaded',  'depth_mm': 10, 'tpeak': None},
    {'storm_type': 'balanced',     'depth_mm': 40, 'tpeak': None},
]

results = engine.predict_batch(test_configs, n_h=N_H, n_w=N_W)
run_and_export( results, test_configs, cfg.output_dir, N_H, N_W, manila_patch_indices, dem_transform, dem_crs, flood_cmap, flood_norm, legend_patches,_barangay_band,)
